# Machine Learning basics with scikit-learn
* we already learned numpy (fast arrays) and pandas (tables) in lesson 19, and how to plot stuff in lesson 20 - now we finally get to the fun part: teaching a computer to actually learn patterns from data
* "machine learning" sounds fancy but the core idea is simple: instead of a human writing exact if/else rules by hand ("if pixel pattern looks like X then its a cat"), we give an algorithm LOTS of examples (input, correct-answer pairs) and it figures out a general pattern/function on its own
* that learned pattern can then predict the answer for NEW inputs it has never seen before - thats the whole point, a model that only works on data it already saw is useless
* the most popular, beginner-friendly ML library in python is **scikit-learn** (`pip install scikit-learn`, usually imported piece by piece, e.g. `from sklearn.tree import DecisionTreeClassifier`) - it has a super consistent API across dozens of algorithms, which is exactly why its still the go-to "classic ML" toolkit today

## supervised vs unsupervised learning
* **supervised learning** - we have labeled examples, meaning we already know the "correct answer" for every training example (e.g. photos of flowers WITH the species name attached) - this is what this whole lesson focuses on
* **unsupervised learning** - we only have the input data, NO labels/correct-answers at all, and the algorithm has to find structure on its own (e.g. "clustering" - grouping similar customers together without ever being told what the groups should be) - just good to know this exists, we wont cover it in depth here
* within supervised learning there are two main flavors:
  * **classification** - predict a CATEGORY/class, e.g. "is this email spam or not spam", "which species is this flower"
  * **regression** - predict a NUMBER, e.g. "what will this house sell for", "how many degrees will it be tomorrow"
* we will do both a classification example and a regression example in this lesson

In [1]:
# scikit-learn ships some small, well-known "toy" datasets directly inside the package - zero download needed, perfect for learning
from sklearn.datasets import load_iris

iris = load_iris()  # this is basically the "hello world" dataset of ML classification, everybody uses it at some point

# .data is a numpy array with the actual measurements (the "features"/"inputs")
print("data shape:", iris.data.shape)  # (150, 4) --> 150 flowers, 4 measurements each
print("first 3 rows of data:\n", iris.data[:3])

# .feature_names tells us what those 4 numbers actually mean
print("\nfeature names:", iris.feature_names)

# .target is a numpy array with the "correct answer" (the label) for every single row, as a number (0, 1 or 2)
print("\nfirst 10 targets:", iris.target[:10])

# .target_names translates those numbers back into human-readable species names
print("target names:", iris.target_names)
# so target 0 means iris.target_names[0] == "setosa", and so on

data shape: (150, 4)
first 3 rows of data:
 [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]

feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

first 10 targets: [0 0 0 0 0 0 0 0 0 0]
target names: ['setosa' 'versicolor' 'virginica']


In [2]:
# lets put this into a pandas DataFrame (remember lesson 19!) to get a nicer, more readable overview
import pandas as pd

iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)  # one column per feature
iris_df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)  # turn the numeric target into readable species names

print(iris_df.head())  # .head() shows the first 5 rows, handy to get a quick feel for the data
print()
print(iris_df["species"].value_counts())  # how many examples do we have of each species? (50 each, nicely balanced)

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

  species  
0  setosa  
1  setosa  
2  setosa  
3  setosa  
4  setosa  

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


## splitting into training and test data
* here is a HUGE, easy-to-make mistake: training a model and then evaluating it on the exact same data it was trained on
* a model can just "memorize" the training data (especially flexible models) and look artificially perfect, while actually being useless on new, unseen data - this is exactly the overfitting problem we discuss later
* the fix: split our data into a **training set** (the model learns from this) and a **test set** (we NEVER show this to the model during training, we only use it afterwards to check how well it really generalizes)
* `train_test_split()` does this split for us, `random_state=42` makes the split reproducible - anyone re-running this notebook gets the exact same split

In [3]:
from sklearn.model_selection import train_test_split

X = iris.data     # by convention, the input features are called "X" (capital, its a 2D matrix) in basically all ML code/tutorials
y = iris.target   # by convention, the labels/target we want to predict are called "y" (lowercase, its a 1D vector)

# test_size=0.2 means 20% of the data goes into the test set, 80% stays for training
# random_state=42 fixes the "randomness" of the split so we get the same result every time we run this
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("training set size:", X_train.shape[0])  # 120 flowers to learn from
print("test set size:", X_test.shape[0])        # 30 flowers held back, only used for evaluation later

training set size: 120
test set size: 30


## training our first classifier
* a **decision tree** is probably the most intuitive ML algorithm there is: it literally learns a sequence of yes/no questions about the features ("is petal length <= 2.5? if yes -> setosa, if no -> ask another question...") until it can decide on a class
* every scikit-learn model follows the exact same simple pattern: create it, `.fit(X_train, y_train)` to train it on the training data, `.predict(X_test)` to get predictions on new data - this consistency is a big reason scikit-learn is so beloved

In [4]:
from sklearn.tree import DecisionTreeClassifier

# create the model - random_state=42 again, some parts of how a tree is built involve randomness (e.g. tie-breaking)
tree_model = DecisionTreeClassifier(random_state=42)

# .fit() is where the actual "learning" happens - the model looks at X_train and y_train and figures out its yes/no questions
tree_model.fit(X_train, y_train)

# .predict() asks the now-trained model to guess the species for flowers it has NEVER seen (our held-out test set)
y_pred = tree_model.predict(X_test)

print("actual species:    ", y_test[:10])
print("predicted species: ", y_pred[:10])
# if the model learned something real, most of these should match up

actual species:     [1 0 2 1 1 0 1 2 1 1]
predicted species:  [1 0 2 1 1 0 1 2 1 1]


## evaluating a classifier
* `accuracy_score` - the simplest metric, just "what fraction of predictions were correct"
* `confusion_matrix` - a small grid showing exactly WHICH classes get mixed up with which - row = actual class, column = predicted class, so cell [i][j] means "how many examples of class i got predicted as class j" (the diagonal = correct predictions, everything off the diagonal = mistakes)
* `classification_report` - gives us precision, recall and f1-score per class:
  * **precision** - of everything the model predicted as class X, how much actually WAS class X (are we crying wolf too often?)
  * **recall** - of everything that actually IS class X, how much did the model correctly catch (are we missing too many?)
  * **f1-score** - a single number that balances precision and recall together, handy when you just want one summary number per class

In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# accuracy - simple percentage of correct predictions on the TEST set (data the model never trained on!)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"test accuracy: {test_accuracy:.2%}")

# confusion matrix - lets us see WHICH species get confused with which, not just an overall percentage
print("\nconfusion matrix:")
print(confusion_matrix(y_test, y_pred))
# rows = actual species (in order setosa, versicolor, virginica), columns = predicted species, same order

# classification_report gives us precision/recall/f1 per class in one readable table
print("\nclassification report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

test accuracy: 100.00%

confusion matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

classification report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## overfitting vs underfitting
* **overfitting** - the model basically memorized the training data, including its random noise/quirks, instead of learning the real underlying pattern - it looks amazing on training data but performs much worse on new data
* **underfitting** - the opposite problem, the model is too simple to even capture the real pattern, so it performs poorly on BOTH the training and the test data
* a classic warning sign for overfitting: compare accuracy on the training set vs the test set - if training accuracy is MUCH higher than test accuracy, the model probably memorized instead of generalizing (a good model should perform roughly similarly on both)

In [6]:
# lets check our decision tree for that warning sign
train_accuracy = accuracy_score(y_train, tree_model.predict(X_train))  # accuracy on data it WAS trained on
print(f"train accuracy: {train_accuracy:.2%}")
print(f"test accuracy:  {test_accuracy:.2%}")

# with iris (a small, "easy", well-separated dataset) both numbers usually end up close/perfect, thats fine here
# but if you ever see e.g. 100% train accuracy and only 70% test accuracy, thats a classic overfitting alarm bell
# common fixes: get more training data, simplify the model (e.g. limit a tree's max_depth), or use regularization (a later-lesson topic)

train accuracy: 100.00%
test accuracy:  100.00%


## a second classifier: k-nearest-neighbors
* **k-nearest-neighbors (KNN)** is another very intuitive algorithm: to predict a new example, it just looks at the "k" closest already-known examples (by distance in feature-space) and takes a majority vote of their labels - no explicit "training" of rules at all, it just remembers the data
* lets train one on the exact same train/test split and compare

In [7]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)  # n_neighbors=5 means "look at the 5 closest known flowers"
knn_model.fit(X_train, y_train)  # KNN "training" is basically just storing the training data

knn_pred = knn_model.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_pred)

print(f"decision tree test accuracy: {test_accuracy:.2%}")
print(f"knn test accuracy:           {knn_accuracy:.2%}")
# on a clean, well-separated dataset like iris, both models usually do great - real-world messier data shows bigger differences between algorithms

decision tree test accuracy: 100.00%
knn test accuracy:           100.00%


## now for regression: predicting a NUMBER
* remember: classification predicts a category, regression predicts a number
* `make_regression()` generates a synthetic (fake, but realistic-looking) regression dataset for us, completely offline, no download needed - great for learning without depending on some external file
* **linear regression** is the simplest regression model: it just tries to find the best-fitting straight line (or hyperplane, with multiple features) through the data

In [8]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# generate synthetic regression data: 200 examples, 1 feature, with some random noise added so its not a perfectly straight line
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

# same idea as before - split into train/test, same random_state for reproducibility
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

linear_model = LinearRegression()
linear_model.fit(X_reg_train, y_reg_train)  # finds the best-fitting line through the training data

y_reg_pred = linear_model.predict(X_reg_test)  # predict numeric values for the test set

# mean_squared_error - average of the squared differences between predicted and actual values (smaller = better, 0 = perfect)
mse = mean_squared_error(y_reg_test, y_reg_pred)
print(f"mean squared error: {mse:.2f}")

# r2_score (R-squared / "coefficient of determination") - how much of the variation in the answer our model actually explains
# 1.0 = perfect predictions, 0 = our model is no better than just always guessing the average value
r2 = r2_score(y_reg_test, y_reg_pred)
print(f"r2 score: {r2:.4f}")

mean squared error: 246.12
r2 score: 0.9681


## quick comparison to other languages
* python's scikit-learn is by far the most widely used general-purpose ML toolkit today, largely riding on python's overall dominance in the data science/ML space (recap lesson 19's numpy/pandas ecosystem) - once your data already lives in numpy arrays/pandas DataFrames, scikit-learn just plugs right in
* **R** has a rich, mature ML ecosystem too (`caret`, and the newer, more modern `tidymodels`) and remains particularly strong in academic statistics, where R has deep historical roots
* **Java** has `Weka`, an older, GUI-friendly ML toolkit that was very popular in university courses - and a lot of enterprise "big data" ML happens via Apache Spark's MLlib, which notably ALSO has a first-class python API, so even "the Java big-data ML world" often ends up being used from python anyway
* **JavaScript** has `TensorFlow.js` and `brain.js` for running/training models directly in a browser or in Node.js - but training large models in JS is rare in practice, JS is far more commonly used to just RUN an already-trained model (this is called "inference") client-side, e.g. a model trained in python and then shipped to run in a web page
* deep learning frameworks (PyTorch, TensorFlow) exist across several languages, but python remains their primary, most complete API by far - which is a big part of why python became the ML lingua franca in the first place

## Exercises
1. load the iris dataset, split it into train/test with `random_state=42`, train a `DecisionTreeClassifier` and print its test accuracy
2. print a `confusion_matrix` for that same model and identify (just by looking at the numbers) which flower species gets confused with which
3. train a `KNeighborsClassifier` on the same train/test split and compare its test accuracy to the decision tree's
4. use `make_regression()` to generate synthetic data, train a `LinearRegression` on it, and print its R² score on the test set

In [9]:
# TODO: solve exercise 1 here (load iris, split with random_state=42, train DecisionTreeClassifier, print test accuracy)


# TODO: solve exercise 2 here (confusion_matrix, figure out which species gets confused with which)


# TODO: solve exercise 3 here (train KNeighborsClassifier on the same split, compare accuracy to the tree)


# TODO: solve exercise 4 here (make_regression + LinearRegression + r2_score on the test set)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [10]:
# sample solution 1
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, r2_score

iris_ex = load_iris()
X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    iris_ex.data, iris_ex.target, test_size=0.2, random_state=42
)
tree_ex = DecisionTreeClassifier(random_state=42)
tree_ex.fit(X_ex_train, y_ex_train)
tree_ex_pred = tree_ex.predict(X_ex_test)
print("exercise 1 - tree test accuracy:", accuracy_score(y_ex_test, tree_ex_pred))

# sample solution 2
cm = confusion_matrix(y_ex_test, tree_ex_pred)
print("\nexercise 2 - confusion matrix:\n", cm)
print("species order:", iris_ex.target_names)
# look at the off-diagonal cells: any non-zero number there tells us exactly which two species got mixed up

# sample solution 3
knn_ex = KNeighborsClassifier(n_neighbors=5)
knn_ex.fit(X_ex_train, y_ex_train)
knn_ex_pred = knn_ex.predict(X_ex_test)
print("\nexercise 3 - tree accuracy:", accuracy_score(y_ex_test, tree_ex_pred),
      "| knn accuracy:", accuracy_score(y_ex_test, knn_ex_pred))

# sample solution 4
X_reg_ex, y_reg_ex = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)
X_reg_ex_train, X_reg_ex_test, y_reg_ex_train, y_reg_ex_test = train_test_split(
    X_reg_ex, y_reg_ex, test_size=0.2, random_state=42
)
lin_ex = LinearRegression()
lin_ex.fit(X_reg_ex_train, y_reg_ex_train)
lin_ex_pred = lin_ex.predict(X_reg_ex_test)
print("\nexercise 4 - r2 score:", r2_score(y_reg_ex_test, lin_ex_pred))

exercise 1 - tree test accuracy: 1.0

exercise 2 - confusion matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
species order: ['setosa' 'versicolor' 'virginica']

exercise 3 - tree accuracy: 1.0 | knn accuracy: 1.0

exercise 4 - r2 score: 0.9680853989031325


* congratulation you finished this lesson

## what we learned
* what machine learning actually is: learning a general pattern from labeled examples instead of hand-writing exact rules
* supervised learning (labeled data - this lesson's focus) vs unsupervised learning (no labels, e.g. clustering)
* classification (predict a category) vs regression (predict a number)
* loading a built-in toy dataset (`load_iris()`), inspecting `.data`/`.feature_names`/`.target`/`.target_names`, and putting it into a pandas DataFrame
* why we always split data into a training set and a held-out test set with `train_test_split()`, and why we never evaluate on training data
* training and using scikit-learn models with the consistent `.fit()`/`.predict()` pattern - `DecisionTreeClassifier` and `KNeighborsClassifier`
* evaluating classifiers with `accuracy_score`, `confusion_matrix` and `classification_report` (precision, recall, f1)
* overfitting (memorizing noise) vs underfitting (too simple to learn the real pattern), and comparing train vs test accuracy as a warning sign
* a basic regression example with synthetic data (`make_regression()`), `LinearRegression`, and evaluating with `mean_squared_error`/`r2_score`